# Day 15: Local Qdrant Instance and Vector Collections

## Core Theory (Just-in-Time)

Vector databases like Qdrant are designed to store, manage, and search high-dimensional vectors. While traditional databases index scalar data, vector databases index embeddings (arrays of floats) using algorithms like HNSW (Hierarchical Navigable Small World) for fast approximate nearest neighbor (ANN) search.

**Why Qdrant?**
Qdrant is written in Rust, offering high performance and memory safety. It supports payload filtering alongside vector search, making it ideal for Retrieval-Augmented Generation (RAG) applications.

**Why Docker for Local Development?**
Running Qdrant via Docker ensures environment consistency between your local machine and production. You don't have to worry about missing dependencies or conflicting database versions on your local OS.

**Vector Dimensions:**
When creating a collection, you must specify the vector's `size` (dimensions) and the `distance` metric (e.g., Cosine, Dot, or Euclidean). The size must exactly match the output dimension of the embedding model you plan to use (e.g., 1536 for OpenAI's `text-embedding-ada-002`).

### Instructions to Run Qdrant locally
Before executing the code, spin up a local instance of Qdrant in your terminal using Docker:
```bash
docker run -d -p 6333:6333 -p 6334:6334 qdrant/qdrant
```
- Port `6333` is for the HTTP API.
- Port `6334` is for the gRPC API (faster, often used in production).

*(Note: The code below uses `location=":memory:"` as a fallback so this notebook can execute natively without requiring the Docker daemon to be running, but in production, you connect to `localhost:6333`.)*
**AI Security Implications:**
1. **PII Protection:** Never embed and store raw PII (Personally Identifiable Information) in your vector database. Sanitize or redact sensitive fields before generating the embedding to prevent data leaks.
2. **Fallbacks:** Always implement a fallback mechanism in case the vector database is unreachable (e.g., returning a graceful error message or relying on a secondary semantic cache).
3. **Prompt Injection:** Be wary of malicious vectors designed to poison your database or return adversarial context to your LLM.


## Code Implementation: Basic

This basic example isolates the core concept of initializing a connection and creating a collection with minimal boilerplate.

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.http import models

# 1. Initialize client (in-memory for local testing)
basic_client = QdrantClient(location=":memory:")

# 2. Create a basic collection
basic_client.create_collection(
    collection_name="basic_collection",
    vectors_config=models.VectorParams(
        size=384,  # e.g., for all-MiniLM-L6-v2
        distance=models.Distance.COSINE
    )
)
print("Basic collection created successfully.")


Basic collection created successfully.


## Code Implementation: Medium

This medium example emphasizes clean OOP principles and state management by wrapping the Qdrant client in a class.

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from typing import Optional

class VectorDatabaseManager:
    """Manages vector database operations with clean state management."""
    
    def __init__(self, in_memory: bool = True):
        self.in_memory = in_memory
        self.client: Optional[QdrantClient] = None
        self.connect()

    def connect(self) -> None:
        """Establishes connection to the vector database."""
        if self.in_memory:
            self.client = QdrantClient(location=":memory:")
        else:
            self.client = QdrantClient(url="http://localhost:6333")
            
    def create_collection(self, name: str, size: int) -> None:
        """Creates a vector collection safely."""
        if not self.client:
            raise RuntimeError("Client is not connected.")
            
        if not self.client.collection_exists(collection_name=name):
            self.client.create_collection(
                collection_name=name,
                vectors_config=models.VectorParams(
                    size=size,
                    distance=models.Distance.COSINE
                )
            )
            print(f"Collection '{name}' created.")
        else:
            print(f"Collection '{name}' already exists.")

if __name__ == "__main__":
    db_manager = VectorDatabaseManager(in_memory=True)
    db_manager.create_collection(name="medium_collection", size=384)


Collection 'medium_collection' created.


## Code Implementation: Advanced

This production-grade implementation features strict type hinting, docstrings, exact import syntax, error handling, and AI security best practices (PII redaction and fallbacks).

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.http.exceptions import UnexpectedResponse
from qdrant_client.http.models import VectorParams, Distance
import logging
import re
from typing import Dict, Any, Optional

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ProductionVectorDB:
    """
    Production-ready Vector Database Manager.
    Demonstrates advanced OOP, AI security (PII redaction), and robust error handling.
    """
    
    def __init__(self, host: str = "localhost", port: int = 6333, in_memory: bool = False) -> None:
        try:
            if in_memory:
                self.client = QdrantClient(location=":memory:")
                logger.info("Initialized in-memory Qdrant client.")
            else:
                self.client = QdrantClient(host=host, port=port, timeout=5.0)
                logger.info(f"Connected to Qdrant at {host}:{port}")
        except Exception as e:
            logger.error(f"Failed to initialize QdrantClient: {e}")
            self.client = None # Fallback mechanism triggers if client is None

    def _redact_pii(self, payload: Dict[str, Any]) -> Dict[str, Any]:
        """
        Sanitizes PII (e.g., email addresses) from metadata before storing.
        """
        sanitized = payload.copy()
        for key, value in sanitized.items():
            if isinstance(value, str):
                # Regex to mask email addresses
                sanitized[key] = re.sub(
                    r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', 
                    '[REDACTED_EMAIL]', 
                    value
                )
        return sanitized

    def initialize_collection(self, collection_name: str, dimension: int) -> bool:
        """
        Creates a collection with robust error handling and fallbacks.
        
        Args:
            collection_name (str): Name of the collection.
            dimension (int): Vector dimension size.
            
        Returns:
            bool: True if successful, False otherwise.
        """
        if self.client is None:
            logger.warning("Fallback triggered: Cannot create collection due to missing DB connection.")
            return False
            
        try:
            if self.client.collection_exists(collection_name=collection_name):
                logger.info(f"Collection '{collection_name}' already exists.")
                return True
                
            self.client.create_collection(
                collection_name=collection_name,
                vectors_config=VectorParams(
                    size=dimension,
                    distance=Distance.COSINE
                )
            )
            logger.info(f"Successfully initialized collection: {collection_name}")
            return True
        except UnexpectedResponse as e:
            logger.error(f"Qdrant API error during collection creation: {e}")
            return False
        except Exception as e:
            logger.error(f"Unexpected error: {e}")
            return False

if __name__ == "__main__":
    # Execute Advanced Example
    prod_db = ProductionVectorDB(in_memory=True)
    prod_db.initialize_collection(collection_name="advanced_collection", dimension=384)
    
    # Test PII redaction
    raw_payload = {"author": "john.doe@example.com", "content": "Confidential data"}
    safe_payload = prod_db._redact_pii(raw_payload)
    logger.info(f"Sanitized Payload: {safe_payload}")


INFO:__main__:Initialized in-memory Qdrant client.


INFO:__main__:Successfully initialized collection: advanced_collection


INFO:__main__:Sanitized Payload: {'author': '[REDACTED_EMAIL]', 'content': 'Confidential data'}


## Practical Lab / Homework

**Task:**
1. Use the initialized `QdrantClient` and collection (`day_15_test_collection`).
2. Upsert three mock vectors into the collection. Each vector must match the `VECTOR_DIMENSION` (384) configured earlier.
3. Include a payload (metadata) for each vector containing at least a `document_id` and a `text_chunk`.
4. Perform a simple nearest-neighbor search using a mock query vector to retrieve the top 2 results.

**Constraint:** 
Write clean, production-ready Python code with strict type hints and docstrings. Do not use pseudo-code.

In [4]:
from qdrant_client import QdrantClient
import random
from qdrant_client.http.models import PointStruct, ScoredPoint
from typing import List

def upsert_mock_vectors(
    client: QdrantClient, 
    collection_name: str, 
    dimension: int
) -> None:
    """
    Upserts a set of mock vectors with payloads into the specified collection.
    
    Args:
        client: The instantiated QdrantClient.
        collection_name: The target collection name.
        dimension: The size of the vectors.
    """
    # Generating 3 mock vectors
    points = [
        PointStruct(
            id=1,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_001", "text_chunk": "Artificial Intelligence is transforming software."}
        ),
        PointStruct(
            id=2,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_002", "text_chunk": "Vector databases are essential for RAG."}
        ),
        PointStruct(
            id=3,
            vector=[random.random() for _ in range(dimension)],
            payload={"document_id": "doc_003", "text_chunk": "LangChain provides abstractions for LLMs."}
        )
    ]
    
    operation_info = client.upsert(
        collection_name=collection_name,
        points=points
    )
    print(f"Upsert operation status: {operation_info.status}")

def search_similar_vectors(
    client: QdrantClient, 
    collection_name: str, 
    query_vector: List[float], 
    limit: int = 2
) -> List[ScoredPoint]:
    """
    Searches the collection for the nearest neighbors to a query vector.
    
    Args:
        client: The instantiated QdrantClient.
        collection_name: The collection to search in.
        query_vector: The vector to compare against.
        limit: The maximum number of results to return.
        
    Returns:
        A list of ScoredPoint objects representing the nearest neighbors.
    """
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=limit
    )
    return search_result.points

# Lab Execution
if __name__ == "__main__":
    # Initialize client for lab
    qdrant_client = QdrantClient(location=":memory:")
    COLLECTION_NAME = "day_15_test_collection"
    VECTOR_DIMENSION = 384
    # Create collection so it exists for upsert
    from qdrant_client.http.models import VectorParams, Distance
    qdrant_client.create_collection(collection_name=COLLECTION_NAME, vectors_config=VectorParams(size=VECTOR_DIMENSION, distance=Distance.COSINE))
    # We reuse the client and collection details from the previous cell
    upsert_mock_vectors(
        client=qdrant_client,
        collection_name=COLLECTION_NAME,
        dimension=VECTOR_DIMENSION
    )
    
    # Create a random query vector matching the dimension
    mock_query = [random.random() for _ in range(VECTOR_DIMENSION)]
    
    results = search_similar_vectors(
        client=qdrant_client,
        collection_name=COLLECTION_NAME,
        query_vector=mock_query,
        limit=2
    )
    
    print("\nSearch Results:")
    for result in results:
        print(f"ID: {result.id}, Score: {result.score:.4f}, Payload: {result.payload}")


Upsert operation status: completed

Search Results:
ID: 2, Score: 0.7679, Payload: {'document_id': 'doc_002', 'text_chunk': 'Vector databases are essential for RAG.'}
ID: 3, Score: 0.7559, Payload: {'document_id': 'doc_003', 'text_chunk': 'LangChain provides abstractions for LLMs.'}


## Common Pitfalls

1. **Dimension Mismatch:** The most frequent error is trying to insert a vector of size 1536 into a collection initialized with size 384. This throws a hard error in Qdrant.
2. **Wrong Distance Metric:** Using Euclidean distance when the embeddings were normalized for Cosine similarity will yield poor retrieval results. Always check your embedding model's documentation for the recommended distance metric.
3. **Missing Indexes in Production:** While small datasets work fine out of the box, production systems require payload indexes on frequently filtered metadata fields to avoid full collection scans.
4. **Ignoring gRPC:** Using the REST API (port 6333) for high-throughput production workloads instead of the faster gRPC interface (port 6334).

## Reference Links

1. [Qdrant Official Documentation](https://qdrant.tech/documentation/)
2. [Qdrant Python Client](https://github.com/qdrant/qdrant-client)
3. [LangChain & Qdrant Integration](https://python.langchain.com/docs/integrations/vectorstores/qdrant)